# Advanced Python descriptors, validation, and geometry
## 18 worked problems • extensive runnable code • assertions and edge cases

**Source basis.** The supplied example introduces an integer descriptor, a `Point2DSequence` descriptor for polygon vertices, `Polygon`/`Triangle`/`Square` subclasses, `collections.abc.Sequence`, and the `__len__`, `__getitem__`, `__iadd__`, `__contains__`, `__eq__`, and `__hash__` special methods. This notebook retains that curriculum and extends it with independently developed advanced problems and solutions.

**Important source limitations addressed here:** returning a writable list lets clients bypass element/length checks; hashing mutable coordinate objects can break `set`/`dict` behavior; and exactly four vertices alone does not guarantee a geometric square. Our solution uses immutable points, tuple-backed vertex storage, validated *whole-value* updates, and a genuinely geometric square validator.

**How to use:** Run **Kernel → Restart & Run All**. Read each challenge before its solution; modify the code to test your own cases. All dependencies are in the Python standard library. All examples use screen coordinates `0 ≤ x ≤ 800` and `0 ≤ y ≤ 400`. The type/length base class enforces only its advertised constraints; geometric conditions are opt-in subclasses. Assertions are teaching tests, not production error-handling.

### Test harness and imports

Expected exceptions should be asserted rather than hidden by `try/except: pass`. The helper checks the exception *type* and an optional message fragment; an unexpected exception still fails the cell.

In [1]:
from collections.abc import Iterable, Mapping, Sequence
from itertools import islice, product
from math import hypot, isclose
import json


def expect_error(error_type, thunk, contains=None):
    """Assert that a zero-argument callable raises a specified exception."""
    try:
        thunk()
    except error_type as exc:
        if contains is not None:
            assert contains in str(exc), (contains, str(exc))
        return str(exc)
    else:
        raise AssertionError(f"Expected {error_type.__name__}, but no error occurred")


print("Test harness ready")

Test harness ready


## Problem 01 — Data descriptor precedence and protocol introspection

**Challenge.** Build a small data descriptor implementing `__set_name__`, `__get__`, `__set__`, and `__delete__`. Demonstrate the difference between class-level access and instance access; determine whether a same-named entry in an instance's `__dict__` can override it.

**Solution design.** A descriptor implementing `__set__` is a **data descriptor**, so it takes precedence over the instance dictionary. Keep stored data under a separate private name, and make deletion explicit.

In [2]:
class LoggedValue:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        instance.__dict__[self.storage_name] = value

    def __delete__(self, instance):
        raise AttributeError(f"Cannot delete {self.name}")


class DescriptorDemo:
    score = LoggedValue()


demo = DescriptorDemo()
demo.score = 7
assert DescriptorDemo.score is DescriptorDemo.__dict__["score"]
assert demo.score == 7
demo.__dict__["score"] = 999  # Deliberately shadow the *public* name.
assert demo.score == 7  # Data descriptor still wins.
expect_error(AttributeError, lambda: delattr(demo, "score"))
print("Class access and data-descriptor precedence verified")

Class access and data-descriptor precedence verified


**Reasoning.** A non-data descriptor (only `__get__`) behaves differently: an ordinary instance attribute can shadow it. A data descriptor is suited to validation because ordinary `obj.attr = value` must go through its `__set__`. Directly changing internal storage or monkey-patching a class is outside this basic invariant model.

## Problem 02 — Reusable, strictly bounded integer descriptor

**Challenge.** Implement `BoundedInt` with optional inclusive lower/upper bounds, validate its configuration, reject `bool` (even though `isinstance(True, int)` holds), report informative errors, and refuse a second assignment. Show descriptor access through the owner class.

**Solution design.** Use `type(value) is int` for this intentionally exact-int API, `None` for unbounded limits, and `__set_name__` to choose a private slot. Disallow reusing one descriptor object under multiple names because its name would be overwritten.

In [3]:
class BoundedInt:
    def __init__(self, min_value=None, max_value=None):
        for label, bound in (("min_value", min_value), ("max_value", max_value)):
            if bound is not None and type(bound) is not int:
                raise TypeError(f"{label} must be an int or None")
        if min_value is not None and max_value is not None and min_value > max_value:
            raise ValueError("min_value cannot exceed max_value")
        self.min_value = min_value
        self.max_value = max_value
        self._binding = None

    def __set_name__(self, owner, name):
        if self._binding is not None and self._binding != (owner, name):
            raise RuntimeError("Use a distinct BoundedInt instance for each attribute")
        self._binding = (owner, name)
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if hasattr(instance, self.storage_name):
            raise AttributeError(f"{self.name} cannot be reassigned")
        if type(value) is not int:
            raise TypeError(f"{self.name} must be an int (not bool)")
        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"{self.name} must be >= {self.min_value}")
        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"{self.name} must be <= {self.max_value}")
        object.__setattr__(instance, self.storage_name, value)

    def __delete__(self, instance):
        raise AttributeError(f"{self.name} cannot be deleted")


expect_error(ValueError, lambda: BoundedInt(6, 4))
expect_error(TypeError, lambda: BoundedInt(True, 4))
assert BoundedInt(0, 2).max_value == 2
print("Validated descriptor configuration")

Validated descriptor configuration


**Subtlety.** An immutable descriptor here is appropriate for point coordinates, but is not a universal replacement for a mutable integer descriptor. A future mutable counter would omit the second-assignment guard.

## Problem 03 — Immutable, hash-safe 2D points

**Challenge.** Implement bounded coordinates, correct `repr`/`str`, value equality, hashing, and immutability. Test invalid type/range, equality of distinct instances, and dictionary lookup with an equivalent point.

**Solution design.** Store coordinates in private slots, route initialization through the descriptors, and block ordinary post-construction assignment. Since the coordinate values cannot normally change, hashing the coordinate tuple is consistent with equality. Return `NotImplemented` for unrelated comparisons.

In [4]:
class Point2D:
    __slots__ = ("_x", "_y")
    x = BoundedInt(min_value=0, max_value=800)
    y = BoundedInt(min_value=0, max_value=400)

    def __init__(self, x, y):
        # object.__setattr__ still invokes a data descriptor named 'x'/'y'.
        object.__setattr__(self, "x", x)
        object.__setattr__(self, "y", y)

    def __setattr__(self, name, value):
        raise AttributeError("Point2D is immutable")

    def __repr__(self):
        return f"Point2D(x={self.x}, y={self.y})"

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __eq__(self, other):
        if not isinstance(other, Point2D):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


origin = Point2D(0, 0)
assert str(origin) == "(0, 0)"
assert repr(origin) == "Point2D(x=0, y=0)"
assert Point2D.x is Point2D.__dict__["x"]
assert Point2D(0, 0) == origin and Point2D(0, 0) is not origin
assert {origin: "found"}[Point2D(0, 0)] == "found"
assert Point2D(1, 1) != "point"
expect_error(TypeError, lambda: Point2D(True, 0))
expect_error(TypeError, lambda: Point2D(0.0, 0))
expect_error(ValueError, lambda: Point2D(-1, 0))
expect_error(ValueError, lambda: Point2D(0, 401))
expect_error(AttributeError, lambda: setattr(origin, "x", 5))
expect_error(AttributeError, lambda: setattr(origin, "_x", 5))
expect_error(AttributeError, lambda: delattr(origin, "y"))
print("Point immutability and hash/equality contract verified")

Point immutability and hash/equality contract verified


**Hashing rule.** If `a == b`, then `hash(a) == hash(b)` must hold, and a hashable object's hash-relevant fields must not change while used as a key. The source example supplies `__hash__` while allowing coordinate reassignment; this revised point avoids that trap. Python's explicit `object.__setattr__` can still bypass ordinary API restrictions: the immutability contract assumes non-adversarial clients.

## Problem 04 — A typed, length-bounded, defensive sequence descriptor

**Challenge.** Implement `VertexSequence` that accepts lists and tuples of `Point2D`, rejects strings, generators and bad elements, stores a snapshot, checks length bounds, and does not return an externally mutable list. Reject deletion and missing initialization rather than silently generating an invalid empty value.

**Solution design.** Validate the entire candidate *before* writing to instance storage. A tuple snapshot prevents mutating the original input list or a returned list to evade validation. A separate descriptor is constructed for every declaring class.

In [5]:
class VertexSequence:
    def __init__(self, min_length=None, max_length=None):
        for label, bound in (("min_length", min_length), ("max_length", max_length)):
            if bound is not None and (type(bound) is not int or bound < 0):
                raise ValueError(f"{label} must be a nonnegative integer or None")
        if min_length is not None and max_length is not None and min_length > max_length:
            raise ValueError("min_length cannot exceed max_length")
        self.min_length = min_length
        self.max_length = max_length
        self._binding = None

    def __set_name__(self, owner, name):
        if self._binding is not None and self._binding != (owner, name):
            raise RuntimeError("Use a distinct VertexSequence for each attribute")
        self._binding = (owner, name)
        self.name = name
        self.storage_name = f"_{name}"

    def _normalize(self, value):
        if not isinstance(value, Sequence) or isinstance(value, (str, bytes, bytearray)):
            raise TypeError(f"{self.name} must be a non-string sequence")
        snapshot = tuple(value)
        if self.min_length is not None and len(snapshot) < self.min_length:
            raise ValueError(f"{self.name} requires at least {self.min_length} items")
        if self.max_length is not None and len(snapshot) > self.max_length:
            raise ValueError(f"{self.name} allows at most {self.max_length} items")
        for index, vertex in enumerate(snapshot):
            if not isinstance(vertex, Point2D):
                raise TypeError(f"{self.name}[{index}] must be Point2D")
        return snapshot

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(f"{self.name} has not been initialized") from None

    def __set__(self, instance, value):
        checked = self._normalize(value)
        instance.__dict__[self.storage_name] = checked

    def __delete__(self, instance):
        raise AttributeError(f"{self.name} cannot be deleted")


expect_error(ValueError, lambda: VertexSequence(min_length=4, max_length=3))
expect_error(ValueError, lambda: VertexSequence(min_length=-1))
print("Sequence descriptor defined")

Sequence descriptor defined


## Problem 05 — Polygon invariants and atomic single-item updates

**Challenge.** Create a `Polygon` of three or more points. Ensure that constructor assignment, reassignment, and `append` validate type and minimum length, reject direct mutation of returned vertices, and preserve old contents after a failed update.

**Solution design.** Use the descriptor for every update (`self.vertices = candidate`), never `self.vertices.append(...)`. The latter bypasses the descriptor if vertices is a mutable list. Start with `Sequence`'s two required primitives to support later protocol exercises.

In [6]:
class Polygon(Sequence):
    vertices = VertexSequence(min_length=3)

    def __init__(self, *vertices):
        self.vertices = vertices

    def _commit(self, candidate):
        """One validation and one storage write; failure leaves old state intact."""
        self.vertices = candidate

    def append(self, point):
        self._commit((*self.vertices, point))

    def __len__(self):
        return len(self.vertices)

    def __getitem__(self, index):
        # Slices return tuples; a 1- or 2-point slice need not be a Polygon.
        return self.vertices[index]

    def __repr__(self):
        return f"{type(self).__name__}({', '.join(map(repr, self.vertices))})"


seed = [Point2D(0, 0), Point2D(2, 0), Point2D(0, 2)]
poly = Polygon(*seed)
assert len(poly) == 3 and isinstance(poly.vertices, tuple)
seed.append(Point2D(3, 3))  # Does not mutate polygon.
assert len(poly) == 3
expect_error(AttributeError, lambda: poly.vertices.append(Point2D(4, 4)))
old = poly.vertices
expect_error(TypeError, lambda: poly.append("not a point"), "[3]")
assert poly.vertices is old
expect_error(ValueError, lambda: setattr(poly, "vertices", (Point2D(0, 0),)))
assert poly.vertices is old
poly.append(Point2D(3, 3))
assert len(poly) == 4
expect_error(AttributeError, lambda: delattr(poly, "vertices"))
print("Atomic append, defensive storage and rollback verified")

Atomic append, defensive storage and rollback verified


**Complexity.** Appending to a tuple-backed polygon copies all `n` vertices: **O(n)** time and space. That cost is intentional for simple invariant-preserving semantics. For very large, frequently edited geometries, use a validated edit buffer and commit once (Problem 09).

## Problem 06 — Subclass constraints without duplicating base methods

**Challenge.** Create `Triangle` (exactly three vertices) and `Quadrilateral` (exactly four). Verify that inherited `Polygon.__init__` and `append` automatically use each subclass's tighter descriptor.

**Solution design.** Override only the `vertices` descriptor; Python's ordinary attribute lookup dispatches `self.vertices` to the most-derived class's descriptor. Observe that a triangle's inherited `append` correctly refuses a fourth vertex.

In [7]:
class Triangle(Polygon):
    vertices = VertexSequence(min_length=3, max_length=3)


class Quadrilateral(Polygon):
    vertices = VertexSequence(min_length=4, max_length=4)


triangle = Triangle(Point2D(0, 0), Point2D(2, 0), Point2D(0, 2))
quad = Quadrilateral(Point2D(0, 0), Point2D(2, 0),
                     Point2D(2, 2), Point2D(0, 2))
assert len(triangle) == 3 and len(quad) == 4
expect_error(ValueError, lambda: triangle.append(Point2D(3, 3)), "at most 3")
expect_error(ValueError, lambda: Triangle(*quad.vertices), "at most 3")
expect_error(ValueError, lambda: Quadrilateral(*triangle.vertices), "at least 4")
assert Polygon.vertices is not Triangle.vertices
assert Triangle.vertices.max_length == 3
print("Descriptor-based subclass specialization verified")

Descriptor-based subclass specialization verified


## Problem 07 — Sequence ABC, slicing, membership and in-place addition

**Challenge.** Show that `Polygon` satisfies `collections.abc.Sequence`, benefits from its `index`/`count` mixins, supports iteration and slicing, and compares contained `Point2D` objects by value. Add a single-point `+=` operation that preserves identity and honors subclass constraints.

**Solution design.** Because `Polygon` already inherits `Sequence` and implements `__len__`/`__getitem__`, the ABC supplies iteration, `__contains__`, `index`, and `count`. Put `__iadd__` in a subclass here to avoid redefining or monkey-patching previously tested classes.

In [8]:
class AugmentedPolygon(Polygon):
    def __iadd__(self, point):
        self.append(point)
        return self


aug = AugmentedPolygon(Point2D(0, 0), Point2D(1, 0), Point2D(0, 1))
assert isinstance(aug, Sequence)
assert list(aug) == list(aug.vertices)
assert aug[0] == Point2D(0, 0)
assert aug[-1] == Point2D(0, 1)
assert aug[:2] == (Point2D(0, 0), Point2D(1, 0))
assert Point2D(0, 0) in aug  # New but equal Point2D instance.
assert aug.index(Point2D(1, 0)) == 1
assert aug.count(Point2D(0, 0)) == 1
original_id = id(aug)
aug += Point2D(1, 1)
assert id(aug) == original_id and len(aug) == 4
expect_error(TypeError, lambda: aug.__iadd__([Point2D(2, 2)]))
print("Sequence mixins and identity-preserving += verified")

Sequence mixins and identity-preserving += verified


**Design choice.** `+=` adds one `Point2D`, not a list of points; bulk editing gets its own explicit API. Silently mixing the two meanings makes type and boundary errors harder to diagnose.

## Problem 08 — Transactional replace, insert, remove and extend

**Challenge.** Add richer editing operations without letting any intermediate state escape. Test an invalid element in a bulk extension, an illegal removal from a triangle, and an invalid replacement index. Require all failed operations to leave the original tuple untouched.

**Solution design.** Build each candidate in a temporary list, then assign the entire candidate through `_commit`. Do not mutate `self.vertices` one element at a time or commit halfway through a batch.

In [9]:
class EditablePolygon(Polygon):
    def replace(self, index, point):
        candidate = list(self.vertices)
        candidate[index] = point  # IndexError occurs before any commit.
        self._commit(candidate)

    def insert(self, index, point):
        candidate = list(self.vertices)
        candidate.insert(index, point)
        self._commit(candidate)

    def remove_at(self, index):
        candidate = list(self.vertices)
        candidate.pop(index)
        self._commit(candidate)

    def extend(self, points):
        # VertexSequence validates that the complete new value is a Sequence.
        candidate = (*self.vertices, *points)
        self._commit(candidate)


class EditableTriangle(EditablePolygon):
    vertices = VertexSequence(min_length=3, max_length=3)


editing = EditablePolygon(Point2D(0, 0), Point2D(1, 0), Point2D(0, 1))
before = editing.vertices
expect_error(TypeError, lambda: editing.extend([Point2D(2, 2), "bad"]))
assert editing.vertices is before
expect_error(IndexError, lambda: editing.replace(99, Point2D(2, 2)))
assert editing.vertices is before
editing.replace(1, Point2D(2, 0))
editing.insert(1, Point2D(1, 1))
editing.extend((Point2D(2, 2), Point2D(3, 3)))
assert len(editing) == 6
editing.remove_at(0)
assert len(editing) == 5
fixed = EditableTriangle(Point2D(0, 0), Point2D(1, 0), Point2D(0, 1))
fixed_before = fixed.vertices
expect_error(ValueError, lambda: fixed.remove_at(0), "at least 3")
assert fixed.vertices is fixed_before
expect_error(ValueError, lambda: fixed.extend([Point2D(2, 2)]), "at most 3")
assert fixed.vertices is fixed_before
print("All edit paths validate and failed commits roll back")

All edit paths validate and failed commits roll back


**API caveat.** `extend` above accepts any Python iterable by unpacking it, whereas `vertices = value` requires a `Sequence`. If you need a strict finite-iterable contract, bound materialization explicitly (Problem 17). An infinite generator must not be passed to unbounded unpacking.

## Problem 09 — Multi-operation atomic transactions

**Challenge.** Apply several edits as one transaction. Support `("append", point)`, `("replace", index, point)`, and `("remove", index)`. A failure in a later operation must not persist earlier operations. Explain why final validation only matters if the transaction never leaks intermediate state.

**Solution design.** Stage all edits in a local list, check operation forms, and call the descriptor exactly once at the end. A final valid result can pass through temporary lengths below the subclass minimum, but only the valid final state is externally published.

In [10]:
class TransactionalPolygon(Polygon):
    def edit(self, operations):
        candidate = list(self.vertices)
        for operation in operations:
            if not isinstance(operation, tuple) or not operation:
                raise ValueError("Operation must be a nonempty tuple")
            kind, *args = operation
            if kind == "append" and len(args) == 1:
                candidate.append(args[0])
            elif kind == "replace" and len(args) == 2:
                index, point = args
                candidate[index] = point
            elif kind == "remove" and len(args) == 1:
                candidate.pop(args[0])
            else:
                raise ValueError(f"Invalid edit operation: {operation!r}")
        self._commit(candidate)  # One descriptor validation + one write.


transactional = TransactionalPolygon(Point2D(0, 0), Point2D(1, 0), Point2D(0, 1))
old = transactional.vertices
expect_error(TypeError, lambda: transactional.edit([
    ("append", Point2D(2, 2)), ("replace", 0, "invalid")
]))
assert transactional.vertices is old
expect_error(IndexError, lambda: transactional.edit([
    ("append", Point2D(2, 2)), ("remove", 99)
]))
assert transactional.vertices is old
transactional.edit([("remove", 0), ("append", Point2D(4, 4)),
                    ("append", Point2D(5, 5))])
assert len(transactional) == 4
print("Multi-operation all-or-nothing transaction verified")

Multi-operation all-or-nothing transaction verified


**Scope.** These operations are atomic with respect to Python-level validation and exceptions in one thread, not automatically thread-safe; `read → build → write` can still lose updates across threads. Synchronization is a separate requirement. The input operations iterable should be finite.

## Problem 10 — Descriptor ownership and direct tampering

**Challenge.** Inspect the owner-visible descriptor objects, demonstrate that a data descriptor ignores an entry with its public name in the instance dictionary, and articulate what direct edits to private storage can bypass.

**Solution design.** Read metadata through `Class.vertices`, never rely on a mutable list returned from instance access, and be explicit that Python object encapsulation is not a security boundary.

In [11]:
d_base = Polygon.vertices
d_tri = Triangle.vertices
d_quad = Quadrilateral.vertices
assert isinstance(d_base, VertexSequence)
assert d_base is not d_tri and d_tri is not d_quad
assert d_base.name == d_tri.name == "vertices"
assert d_tri._binding == (Triangle, "vertices")
probe = Polygon(Point2D(0, 0), Point2D(1, 0), Point2D(0, 1))
original = probe.vertices
probe.__dict__["vertices"] = ("fake public entry",)
assert probe.vertices is original  # Data-descriptor lookup wins.
assert probe.__dict__["_vertices"] is original
print("Owner access and lookup precedence verified")

Owner access and lookup precedence verified


**Security / correctness boundary.** Client code that deliberately writes `probe.__dict__["_vertices"]` can subvert a dictionary-backed descriptor. This design protects ordinary public operations and accidental aliasing, not hostile reflection. Truly constrained external inputs still need validation at every serialization/API boundary; consider restricted storage or a different language/runtime when enforcement against adversarial code is required.

## Problem 11 — Shoelace area, perimeter and bounding box

**Challenge.** Implement signed area, absolute area, perimeter and axis-aligned bounding box. Verify a triangle numerically, show that reversed winding flips signed area but not absolute area, and specify the effect of a collinear triangle.

**Solution design.** Pair every vertex with the next, wrapping around to the first. Signed area is `½ Σ(xᵢyᵢ₊₁ − xᵢ₊₁yᵢ)`; perimeter sums Euclidean edge lengths. This problem adds calculations, not new geometric-validity requirements.

In [12]:
class GeometricPolygon(Polygon):
    def signed_area(self):
        vertices = self.vertices
        double_area = sum(
            a.x * b.y - b.x * a.y
            for a, b in zip(vertices, vertices[1:] + vertices[:1])
        )
        return double_area / 2

    def area(self):
        return abs(self.signed_area())

    def perimeter(self):
        vertices = self.vertices
        return sum(
            hypot(a.x - b.x, a.y - b.y)
            for a, b in zip(vertices, vertices[1:] + vertices[:1])
        )

    def bounding_box(self):
        return (min(p.x for p in self.vertices), min(p.y for p in self.vertices),
                max(p.x for p in self.vertices), max(p.y for p in self.vertices))


right_tri = GeometricPolygon(Point2D(0, 0), Point2D(3, 0), Point2D(0, 4))
reversed_tri = GeometricPolygon(*reversed(right_tri.vertices))
assert right_tri.signed_area() == 6
assert reversed_tri.signed_area() == -6
assert right_tri.area() == reversed_tri.area() == 6
assert isclose(right_tri.perimeter(), 12.0)
assert right_tri.bounding_box() == (0, 0, 3, 4)
collinear = GeometricPolygon(Point2D(0, 0), Point2D(1, 1), Point2D(2, 2))
assert collinear.area() == 0  # This base type intentionally allows degeneracy.
print("Shoelace, winding, perimeter and bounds verified")

Shoelace, winding, perimeter and bounds verified


**Assumption.** The shoelace formula computes algebraic (signed) area for the *given vertex order*. For self-intersecting polygonal walks it is not the same as the total area of the visual union of regions; validating simplicity is a separate problem.

## Problem 12 — A real square validator (not merely four points)

**Challenge.** Validate that four vertices, **in boundary order**, form a non-degenerate square: four equal nonzero edges, four right angles, consistent turns, and nonzero area. Reject an ordinary rectangle, a rhombus, and a bow-tie ordering. Ensure replacing a valid square with an invalid sequence fails atomically.

**Solution design.** Subclass the sequence descriptor and override `_normalize`, so every assignment invokes geometric validation. Validate vectors using integer dot/cross products and squared lengths: no floating-point tolerance is necessary with integer coordinates.

In [13]:
class SquareVertices(VertexSequence):
    def __init__(self):
        super().__init__(min_length=4, max_length=4)

    def _normalize(self, value):
        pts = super()._normalize(value)
        edges = [
            (pts[(i + 1) % 4].x - pts[i].x,
             pts[(i + 1) % 4].y - pts[i].y)
            for i in range(4)
        ]
        lengths_squared = [dx * dx + dy * dy for dx, dy in edges]
        if lengths_squared[0] == 0 or len(set(lengths_squared)) != 1:
            raise ValueError("A square needs four equal, nonzero sides")
        turns = []
        for i in range(4):
            ax, ay = edges[i]
            bx, by = edges[(i + 1) % 4]
            if ax * bx + ay * by != 0:
                raise ValueError("Adjacent square edges must be perpendicular")
            turns.append(ax * by - ay * bx)
        if any(turn == 0 or (turn > 0) != (turns[0] > 0) for turn in turns):
            raise ValueError("Square edges must turn consistently")
        return pts


class Square(GeometricPolygon):
    vertices = SquareVertices()


square = Square(Point2D(0, 0), Point2D(2, 0),
                Point2D(2, 2), Point2D(0, 2))
assert square.area() == 4 and square.perimeter() == 8
assert square.bounding_box() == (0, 0, 2, 2)
expect_error(ValueError, lambda: Square(Point2D(0, 0), Point2D(3, 0),
                                        Point2D(3, 2), Point2D(0, 2)))
expect_error(ValueError, lambda: Square(Point2D(1, 0), Point2D(2, 1),
                                        Point2D(1, 2), Point2D(0, 0)))
expect_error(ValueError, lambda: Square(Point2D(0, 0), Point2D(2, 2),
                                        Point2D(2, 0), Point2D(0, 2)))
square_before = square.vertices
expect_error(ValueError, lambda: setattr(square, "vertices", (
    Point2D(0, 0), Point2D(3, 0), Point2D(3, 2), Point2D(0, 2))))
assert square.vertices is square_before
expect_error(ValueError, lambda: square.append(Point2D(5, 5)), "at most 4")
print("True square geometry and failure-atomic reassignment verified")

True square geometry and failure-atomic reassignment verified


**Terminology correction.** In the supplied lesson, `Square` enforces only the vertex count of four; it is closer to a four-vertex polygon type. Here `Square` additionally checks geometry. The boundary-order precondition is important: an unordered collection of four square corners is not automatically a valid polygon walk.

## Problem 13 — Non-degenerate triangles through an overridable validator

**Challenge.** Keep the ordinary `Triangle` as a three-point *typed container* but introduce `StrictTriangle`, which rejects repeated points and collinearity. Verify a counterclockwise and clockwise triangle are both allowed.

**Solution design.** Compute an exact cross product for three integer-coordinate points; its nonzero value is equivalent to positive absolute triangle area. Put the rule in the descriptor so constructor calls and later whole-sequence assignments obey it.

In [14]:
class NonDegenerateTriangleVertices(VertexSequence):
    def __init__(self):
        super().__init__(min_length=3, max_length=3)

    def _normalize(self, value):
        pts = super()._normalize(value)
        a, b, c = pts
        cross = ((b.x - a.x) * (c.y - a.y)
                 - (b.y - a.y) * (c.x - a.x))
        if cross == 0:
            raise ValueError("Triangle must have nonzero area")
        return pts


class StrictTriangle(GeometricPolygon):
    vertices = NonDegenerateTriangleVertices()


strict = StrictTriangle(Point2D(0, 0), Point2D(3, 0), Point2D(0, 4))
assert strict.area() == 6
assert StrictTriangle(*reversed(strict.vertices)).signed_area() == -6
expect_error(ValueError, lambda: StrictTriangle(Point2D(0, 0),
              Point2D(1, 1), Point2D(2, 2)), "nonzero area")
expect_error(ValueError, lambda: StrictTriangle(Point2D(0, 0),
              Point2D(0, 0), Point2D(2, 2)), "nonzero area")
assert len(Triangle(Point2D(0, 0), Point2D(1, 1), Point2D(2, 2))) == 3
print("Opt-in geometric validity vs basic cardinality made explicit")

Opt-in geometric validity vs basic cardinality made explicit


## Problem 14 — Safe JSON serialization and deserialization

**Challenge.** Serialize the polygon type and its coordinates to a JSON-compatible dictionary, then reconstruct it. Preserve subclass behavior and reject unrecognized types, extra fields, malformed coordinates, and invalid geometry. Verify a JSON round-trip.

**Solution design.** Never instantiate types from arbitrary input strings using `eval` or `globals()`. Use a fixed allowlist registry and pass untrusted coordinates back through `Point2D` and the descriptor-backed constructors.

In [15]:
POLYGON_TYPES = {
    "Polygon": Polygon,
    "Triangle": Triangle,
    "Quadrilateral": Quadrilateral,
    "Square": Square,
    "StrictTriangle": StrictTriangle,
}


def polygon_to_dict(polygon):
    kind = type(polygon).__name__
    if kind not in POLYGON_TYPES or type(polygon) is not POLYGON_TYPES[kind]:
        raise TypeError("Unregistered polygon type")
    return {"kind": kind, "vertices": [[p.x, p.y] for p in polygon.vertices]}


def polygon_from_dict(payload):
    if not isinstance(payload, Mapping) or set(payload) != {"kind", "vertices"}:
        raise ValueError("Expected exactly 'kind' and 'vertices'")
    kind = payload["kind"]
    if type(kind) is not str or kind not in POLYGON_TYPES:
        raise ValueError("Unknown polygon kind")
    raw_vertices = payload["vertices"]
    if not isinstance(raw_vertices, (list, tuple)):
        raise TypeError("vertices must be a JSON-style array")
    points = []
    for index, coordinates in enumerate(raw_vertices):
        if not isinstance(coordinates, (list, tuple)) or len(coordinates) != 2:
            raise ValueError(f"vertices[{index}] must have two coordinates")
        points.append(Point2D(*coordinates))
    return POLYGON_TYPES[kind](*points)


serialized = json.dumps(polygon_to_dict(square), sort_keys=True)
restored = polygon_from_dict(json.loads(serialized))
assert type(restored) is Square and restored.vertices == square.vertices
expect_error(ValueError, lambda: polygon_from_dict({
    "kind": "UntrustedClass", "vertices": []}), "Unknown")
expect_error(ValueError, lambda: polygon_from_dict({
    "kind": "Polygon", "vertices": [], "extra": 3}))
expect_error(TypeError, lambda: polygon_from_dict({
    "kind": "Triangle", "vertices": [[True, 0], [2, 0], [0, 2]]}))
expect_error(ValueError, lambda: polygon_from_dict({
    "kind": "Square", "vertices": [[0, 0], [3, 0], [3, 2], [0, 2]]}))
print("Round-trip:", serialized)

Round-trip: {"kind": "Square", "vertices": [[0, 0], [2, 0], [2, 2], [0, 2]]}


**Boundary policy.** JSON numbers such as `1.0` remain floats, which this exact-integer model intentionally rejects. A real API should document whether conversions or stricter schema checks are desired; silently truncating is usually undesirable for coordinates.

## Problem 15 — Deterministic geometry caching with correct invalidation

**Challenge.** Cache the bounding box so repeated reads do not rescan all vertices. Only invalidate after a successful commit; if a bad update raises, both the vertex tuple and valid cached value must survive.

**Solution design.** Override the existing `_commit` hook, not every editing method. Perform superclass validation/write first, then clear the cache. Track recomputations to make the optimization verifiable.

In [16]:
class CachedGeometricPolygon(GeometricPolygon):
    def __init__(self, *vertices):
        self._bbox_cache = None
        self.bbox_computations = 0
        super().__init__(*vertices)

    def _commit(self, candidate):
        super()._commit(candidate)  # Validation may raise; do not invalidate then.
        self._bbox_cache = None

    def bounding_box(self):
        if self._bbox_cache is None:
            self._bbox_cache = super().bounding_box()
            self.bbox_computations += 1
        return self._bbox_cache


cached = CachedGeometricPolygon(Point2D(0, 0), Point2D(2, 0), Point2D(0, 2))
assert cached.bounding_box() == (0, 0, 2, 2)
assert cached.bounding_box() == (0, 0, 2, 2)
assert cached.bbox_computations == 1
old_box, old_vertices = cached.bounding_box(), cached.vertices
expect_error(TypeError, lambda: cached.append("bad"))
assert cached.vertices is old_vertices and cached.bounding_box() is old_box
assert cached.bbox_computations == 1
cached.append(Point2D(5, 1))
assert cached.bounding_box() == (0, 0, 5, 2)
assert cached.bbox_computations == 2
print("Successful commits invalidate, failed commits preserve the cache")

Successful commits invalidate, failed commits preserve the cache


**Scope.** Call `cached.append` or `_commit` to edit; direct `cached.vertices = ...` bypasses this subclass's cache invalidation hook. A fully general cached model would move invalidation into a descriptor callback (or forbid direct assignment behind a property) so *every* supported edit path invalidates. Problem 16 implements the stronger version.

## Problem 16 — Complete cache invalidation for direct descriptor assignment

**Challenge.** Repair the previous cache design so *all* successful `vertices` assignments invalidate it, including direct assignments. Failed assignments must leave the cache intact. Keep the solution reusable without replacing the descriptor contract.

**Solution design.** Override the descriptor's `__set__`: validate and store first, then call a model-specific invalidation method. The cache is initialized before `Polygon.__init__` assigns vertices. No invalidation occurs if superclass validation raises.

In [17]:
class CacheAwareVertices(VertexSequence):
    def __set__(self, instance, value):
        super().__set__(instance, value)
        instance._invalidate_bbox()


class FullyCachedPolygon(GeometricPolygon):
    vertices = CacheAwareVertices(min_length=3)

    def __init__(self, *vertices):
        self._bbox_cache = None
        self.bbox_computations = 0
        super().__init__(*vertices)

    def _invalidate_bbox(self):
        self._bbox_cache = None

    def bounding_box(self):
        if self._bbox_cache is None:
            self._bbox_cache = super().bounding_box()
            self.bbox_computations += 1
        return self._bbox_cache


full_cache = FullyCachedPolygon(Point2D(0, 0), Point2D(1, 0), Point2D(0, 1))
assert full_cache.bounding_box() == (0, 0, 1, 1)
assert full_cache.bbox_computations == 1
full_cache.vertices = (Point2D(0, 0), Point2D(7, 0), Point2D(0, 7))
assert full_cache.bounding_box() == (0, 0, 7, 7)
assert full_cache.bbox_computations == 2
stable_vertices = full_cache.vertices
stable_box = full_cache.bounding_box()
expect_error(TypeError, lambda: setattr(full_cache, "vertices", (
    Point2D(0, 0), "bad", Point2D(0, 1))))
assert full_cache.vertices is stable_vertices
assert full_cache.bounding_box() is stable_box
assert full_cache.bbox_computations == 2
full_cache.append(Point2D(8, 8))
assert full_cache.bounding_box() == (0, 0, 8, 8)
assert full_cache.bbox_computations == 3
print("Cache handles both direct assignment and inherited editing")

Cache handles both direct assignment and inherited editing


## Problem 17 — Safely materializing generators under a maximum length

**Challenge.** The descriptor explicitly requires a sequence, yet an upstream source may provide a finite iterator. Write an adapter that converts an iterable to a tuple while never consuming more than `max_items + 1` elements. Detect a nonterminating source in bounded time, and leave element validation to the descriptor.

**Solution design.** Check a positive integral cap, use `itertools.islice` to bound reads, reject oversized input, then instantiate the polygon through its regular validating constructor. Limit conversion at the boundary; do not silently relax the descriptor to accept all iterables.

In [18]:
def bounded_materialize(values, max_items):
    if type(max_items) is not int or max_items < 0:
        raise ValueError("max_items must be a nonnegative int")
    if not isinstance(values, Iterable):
        raise TypeError("values must be iterable")
    snapshot = tuple(islice(values, max_items + 1))
    if len(snapshot) > max_items:
        raise ValueError(f"More than {max_items} items")
    return snapshot


stream = (Point2D(i, i) for i in range(3))
items = bounded_materialize(stream, 3)
assert isinstance(items, tuple) and len(Polygon(*items)) == 3
expect_error(ValueError, lambda: bounded_materialize(
    (Point2D(0, 0) for _ in iter(int, 1)), 4), "More than 4")
expect_error(TypeError, lambda: Polygon(*bounded_materialize(123, 10)))
expect_error(TypeError, lambda: setattr(poly, "vertices", (p for p in items)))
expect_error(ValueError, lambda: bounded_materialize(items, True))
print("Iterators are bounded before descriptor validation")

Iterators are bounded before descriptor validation


**Resource note.** The adapter bounds item *count*, not memory per item or upstream computation per `next()` call. External untrusted streams may require time, size, and serialization limits too.

## Problem 18 — Exhaustive small-grid and metamorphic tests

**Challenge.** Explore all triples of the nine integer points in a 3×3 grid. For each, verify signed area reverses sign under reversed order, translation leaves area unchanged, absolute area is nonnegative, and coordinates remain valid after translation. Check value equality and hash consistency over the entire grid.

**Solution design.** A finite exhaustive test is more reproducible than an unseeded random test. Geometry gives metamorphic properties: translation does not change area and reversing vertex order changes only its sign.

In [19]:
grid = [Point2D(x, y) for x, y in product(range(3), repeat=2)]
checked = 0
for a, b, c in product(grid, repeat=3):
    candidate = GeometricPolygon(a, b, c)
    mirror = GeometricPolygon(c, b, a)
    shifted = GeometricPolygon(*(Point2D(p.x + 10, p.y + 10)
                                 for p in candidate))
    assert candidate.signed_area() == -mirror.signed_area()
    assert candidate.signed_area() == shifted.signed_area()
    assert candidate.area() >= 0
    assert candidate.area() == abs(candidate.signed_area())
    checked += 1
for p in grid:
    twin = Point2D(p.x, p.y)
    assert p == twin and hash(p) == hash(twin)
assert checked == 729
print(f"PASS: {checked} exhaustive geometry cases + 9 hash/equality cases")

PASS: 729 exhaustive geometry cases + 9 hash/equality cases


---
## Bonus challenge A — Why `__hash__` plus mutable coordinates fails

**Problem.** The source example introduces `Point2D.__hash__ = hash((x, y))` while the coordinates remain assignable. Construct a *separate intentionally broken* mutable class, place it in a set, mutate a hash-relevant attribute, and explain the resulting lookup mismatch. Do **not** use mutable hashable points in production.

**Worked solution.** This demonstration is isolated from the safe `Point2D` implementation above.

In [20]:
class BadMutablePoint:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        if not isinstance(other, BadMutablePoint):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


bad = BadMutablePoint(0, 0)
old_hash = hash(bad)
bad.x = 1
assert hash(bad) != old_hash  # For these tuples in CPython/Python, hashes differ.
# The original hash bucket no longer corresponds to this object's new hash.
# Avoid asserting membership results: set probing is an implementation detail.
print("Broken contract: hash changed after mutation; use immutable points instead")

Broken contract: hash changed after mutation; use immutable points instead


**Takeaway.** A hashable value object must not change its equality/hash-relevant state while inside hash-based containers. Alternatively, define `__hash__ = None` on a deliberately mutable, value-equal class.

## Bonus challenge B — Safe API contracts and what they do not guarantee

**Question.** Does `Square` prove that an arbitrary set of four points forms a square? Does `VertexSequence` protect against writes into `__dict__`? Are edits thread-safe? Does shoelace area imply a simple polygon?

**Solution.** No: the square validator assumes an **ordered boundary walk**; no: dictionary/private-attribute tampering is outside the public contract; no: the read–modify–write edit API needs synchronization for concurrent writes; no: a self-intersecting polygon requires separate topological validation. For a production package, document these as explicit preconditions rather than implying stronger guarantees.

## Final integrated verification

This final cell cross-checks behavior across descriptor dispatch, geometry, serialization, rollback, immutable value objects, and cached models. If **Restart & Run All** reaches `ALL CHECKS PASSED`, the notebook has executed end-to-end with no failing assertions.

In [21]:
def final_smoke_test():
    a, b, c, d = (Point2D(0, 0), Point2D(4, 0),
                  Point2D(4, 4), Point2D(0, 4))
    s = Square(a, b, c, d)
    assert s.area() == 16 and s.perimeter() == 16
    assert type(polygon_from_dict(json.loads(json.dumps(polygon_to_dict(s))))) is Square
    assert Point2D(4, 4) in s and len(s[:2]) == 2
    before = s.vertices
    expect_error(ValueError, lambda: s.append(Point2D(5, 5)))
    assert s.vertices is before
    cached2 = FullyCachedPolygon(a, b, c)
    assert cached2.bounding_box() == (0, 0, 4, 4)
    cached2.vertices = (a, c, d)
    assert cached2.bounding_box() == (0, 0, 4, 4)
    assert cached2.bbox_computations == 2
    assert len({a, Point2D(0, 0)}) == 1
    return True


assert final_smoke_test()
print("ALL CHECKS PASSED — 18 advanced problems, 2 bonus challenges, and final integration")

ALL CHECKS PASSED — 18 advanced problems, 2 bonus challenges, and final integration


## Review and further practice

1. Implement an immutable `Polygon` whose editing methods return new objects instead of mutating the receiver. How does this change cache and concurrency behavior?
2. Add an `__init_subclass__` check that confirms subclass descriptor bounds are compatible with a declared `vertex_count`.
3. For any four unique points in arbitrary order, find a correct boundary ordering or reject them; keep this separate from `SquareVertices`' boundary-order contract.
4. Detect non-adjacent polygon edge intersections using orientation and on-segment predicates; reject self-intersecting walks before reporting geometric interior area.
5. Specify a thread-safe polygon mutation API and its lock granularity. Explain the trade-off between holding the lock during validation versus optimistic version checks.
6. Explore using `typing.Generic`, `collections.abc.MutableSequence`, or controlled mutation proxies for a reusable typed container. Do not return the raw list if callers must not bypass validation.

**Practical checklist:** class access returns the descriptor; bool is not accepted as an int; range, type, and length validation are explicit; all public writes validate before publication; points are hash-stable; geometric claims match the checks actually implemented; deserialization uses an allowlist; tests cover failures and rollback; performance and concurrency assumptions are documented.